# 00 — BEDOL data audit and frozen chronological split


**Purpose.** Verify the public six-site CSV release before any model is trained. This notebook checks the English-only filenames, SHA-256 hashes, row counts, target completeness, quality flags, and the fixed train/validation/test chronology. It writes only compact audit CSV files; it does not train a model.


In [ ]:
from pathlib import Path
import sys, hashlib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT/"src"))
from bedol_repro import load_public_data, audit_public_data

DATA_DIR = ROOT/"data"
CONFIG_DIR = ROOT/"configs"
OUT = ROOT/"reproduction_outputs"
OUT.mkdir(exist_ok=True)

manifest, frames = load_public_data(DATA_DIR, CONFIG_DIR/"site_manifest.csv")
splits = pd.read_csv(CONFIG_DIR/"chronological_splits.csv")
print("Loaded sites:", list(frames))


In [ ]:
# Verify file hashes against the release manifest.
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

audit = audit_public_data(frames, manifest)
audit["sha256_loaded"] = [sha256(DATA_DIR/f) for f in audit.filename]
audit = audit.merge(manifest[["filename","sha256"]], on="filename", how="left")
audit["sha256_match"] = audit.sha256_loaded.eq(audit.sha256)
audit.to_csv(OUT/"00_data_audit.csv", index=False)
display(audit)
assert audit.sha256_match.all(), "At least one public CSV does not match the release manifest."
assert (audit.target_missing == 0).all(), "The filled target contains missing values."


In [ ]:
# Verify the frozen chronological split.
split_audit = []
for site, df in frames.items():
    s = splits.loc[splits.site_id.eq(site)].iloc[0]
    for part, start, end in [
        ("validation", pd.Timestamp(s.val_start), pd.Timestamp(s.val_end)),
        ("test", pd.Timestamp(s.test_start), pd.Timestamp(s.test_end)),
    ]:
        n = int(((df.ds >= start) & (df.ds <= end)).sum())
        split_audit.append({"site_id":site,"split":part,"start":start,"end":end,"hourly_rows":n})

split_audit = pd.DataFrame(split_audit)
split_audit.to_csv(OUT/"00_split_audit.csv", index=False)
display(split_audit)
assert (split_audit.hourly_rows == 8760).all(), "Validation and test should each contain 8,760 hourly rows per site."
print("Audit passed. Proceed to Notebook 01.")
